# Map of Indian districts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themapsdaily/india-choropleth-maps/blob/main/notebooks/02_district_map.ipynb)

**What you will make:** a colour-shaded map of districts with state borders drawn on top, plus a colour bar and ranked bar charts for the highest and lowest districts.

**What you need:**

1. A boundary file for this map (GeoJSON, GeoPackage or zipped shapefile) with the columns `dt_nm` (district) and `st_nm` (state). We do not ship boundary files. See the README for where to find one.
2. A table with the columns `DISTRICT`, `STATE` and `value`. A made-up example is in `examples/sample_district_values.csv`. District names repeat across states, so every row needs the state too.

## 1. Set up

In Colab this installs the package. On your own computer run `pip install indiamaps` once, or install it from a clone of this repository.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "indiamaps @ git+https://github.com/themapsdaily/india-choropleth-maps.git"

## 2. Settings

Everything you are likely to change is in this one cell.

In [ ]:
import os
from indiamaps import colors

REGION = "districts"
BOUNDARY = os.environ.get("INDIAMAPS_BOUNDARY", "india_districts.geojson")   # your boundary file
TABLE = os.environ.get("INDIAMAPS_TABLE", "examples/sample_district_values.csv")  # your numbers
COLOR = colors.dodger_blue   # try colors.persian_green, colors.radical_red, or any [R, G, B]

THEME = "Sample values"      # what the numbers are, for your own reference
DIGITS = 0                   # decimals shown on the map
PREFIX, SUFFIX = "", "%"     # shown before and after each number
VMIN, VMAX = None, None      # None = use the lowest and highest value in your table
FONT = None                  # path to a .ttf or .otf file, or None for the default font

## 3. Choose your colours

The colour scale runs from white to the colour you chose. To blend through several colours, pass them all: `build_custom_continuous_cmap(colors.white, colors.yellow, colors.radical_red)`.

Colours you can use: white, persian_green, light_green, citrus_green, olive_green, yellow, light_yellow, golden_yellow, dark_orange, radical_red, scarlet_red, pink, purple, medium_purple, royal_blue, cornflower_blue, sky_blue, dodger_blue, turquoise_blue, pacific_blue, navy_blue and grey.

In [ ]:
import indiamaps as im

cmap = im.build_custom_continuous_cmap(colors.white, COLOR)

## 4. Load your boundary file

In Colab, run the next cell if you want to upload the file from your computer. Skip it if the file is already in the folder.

In [ ]:
if not os.path.exists(BOUNDARY):
    try:
        from google.colab import files
        uploaded = files.upload()
        BOUNDARY = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put your boundary file at {BOUNDARY} or set INDIAMAPS_BOUNDARY")

spec = im.get_region(REGION)
boundaries = im.load_boundaries(BOUNDARY, name_col=spec.name_col)
boundaries.head()

## 5. Load your numbers and join them to the map

The join matches names ignoring capital letters and punctuation, and prints a warning for any name in either file that found no match.

In [ ]:
import pandas as pd

table = pd.read_csv(TABLE, comment="#")
merged = im.join_region(REGION, boundaries, table, value_col="value")
merged[["value"]].describe()

## 6. Set the scale

By default the scale runs from the lowest to the highest value. To compare maps side by side, set `VMIN` and `VMAX` by hand in the settings cell.

In [ ]:
vmin, vmax = im.default_limits(merged["value"], VMIN, VMAX)
vmin, vmax

## 7. Draw the map

Regions with no number are drawn in grey. Numbers on the map and borders on top follow the defaults for this kind of map.

In [ ]:
fig, ax = im.plot_region_map(
    REGION, merged, cmap, vmin, vmax,
    boundaries=boundaries, digits=DIGITS, prefix=PREFIX, suffix=SUFFIX, font_path=FONT,
)
fig

## 8. Colour bar and ranked bar chart

In [ ]:
cb = im.colorbar(vmin, vmax, cmap)
chart_fig, chart_ax = im.region_bar_chart(
    REGION, merged, COLOR, digits=DIGITS, prefix=PREFIX, suffix=SUFFIX, font_path=FONT,
)
chart_fig

## 9. Save your images

The map is saved with a transparent background so it can sit on a social card or a slide.

In [ ]:
os.makedirs("output", exist_ok=True)
im.save_figure(fig, "output/map.png", dpi=300, transparent=True)
im.save_figure(cb, "output/colorbar.png", dpi=300)
im.save_figure(chart_fig, "output/bar_chart.png", dpi=300, transparent=True, tight=True)
sorted(os.listdir("output"))

## 10. Optional: a social card

Puts the map, colour bar and your words on one image. Fonts and a background picture are optional: pass your own files, or leave them out for a plain white card with the default font.

In [ ]:
text = im.CardText(
    header_1=THEME.upper(),
    scale_min=f"{PREFIX}{vmin:g}{SUFFIX}", scale_max=f"{PREFIX}{vmax:g}{SUFFIX}",
    footer_2="Source: add your source here",
    footer_4="Areas in grey, if any, had no data.",
)
card = im.map_card("output/map.png", "output/colorbar.png", text, layout="district")
im.save_card(card, "output/card.png")
card.resize((540, 675))